In [1]:
# Parameters
update_sort_set = False
download_set = True


In [2]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import os
import io
import time
from datetime import datetime
import re

# ==========================================
# ⚙️ ตั้งค่าเริ่มต้น (Configuration)
# ==========================================
FOLDER_LIST = [
    "1BInVYvIoKW9jpG135ubNTlLdEakR1IKG"
]

DOWNLOAD_FOLDER_NAME = "ราคากลาง"
FORMULA_A1 = '=TEXT(TODAY(), "[$-th-TH]mmm e")'

# ==========================================
# 🛠️ ฟังก์ชันช่วยเหลือสำหรับจัดการชื่อไฟล์
# ==========================================

def get_thai_month_year():
    """สร้างข้อความเดือนย่อ และ ปี พ.ศ. สองหลัก เช่น 'ก.พ. 69'"""
    thai_months = ["", "ม.ค.", "ก.พ.", "มี.ค.", "เม.ย.", "พ.ค.", "มิ.ย.", "ก.ค.", "ส.ค.", "ก.ย.", "ต.ค.", "พ.ย.", "ธ.ค."]
    now = datetime.now()
    month_str = thai_months[now.month]
    year_str = str(now.year + 543)[-2:]
    return f"{month_str} {year_str}"

def generate_new_filename(current_name):
    """เปลี่ยนเฉพาะส่วนเดือน/ปี ในชื่อไฟล์ โดยรักษาโครงสร้างเดิมไว้"""
    target_date = get_thai_month_year()
    # ตรรกะ: ถ้าเจอ "_" ให้เดาว่าส่วนกลางคือวันที่ เช่น "ชื่อ_ม.ค. 69_gs"
    if "_" in current_name:
        parts = current_name.split("_")
        if len(parts) >= 3:
            return f"{parts[0]}_{target_date}_{parts[2]}"
    return current_name # ถ้าไม่เข้าเงื่อนไข คืนชื่อเดิม
def generate_new_filename(current_name):
    """เปลี่ยนเดือน/ปี ในชื่อไฟล์อัตโนมัติ ไม่ว่าจะอยู่ส่วนไหนของชื่อ"""
    target_date = get_thai_month_year() # เช่น 'มี.ค. 69'
    
    # รูปแบบ: เดือนย่อตามด้วยปี 2 หลัก (เช่น ม.ค. 68, ก.พ. 68)
    # \w{1,4}\.?\s?\d{2} คือการหาตัวอักษร(เดือน) ตามด้วยจุด(ถ้ามี) และปี 2 หลัก
    pattern = r"(ม\.ค\.|ก\.พ\.|มี\.ค\.|เม\.ย\.|พ\.ค\.|มิ\.ย\.|ก\.ค\.|ส\.ค\.|ก\.ย\.|ต\.ค\.|พ\.ย\.|ธ\.ค\.)\s?\d{2}"
    
    if re.search(pattern, current_name):
        # ถ้าเจอเดือนเก่า ให้เปลี่ยนเป็นเดือนปัจจุบัน
        new_name = re.sub(pattern, target_date, current_name)
        return new_name
    
    # ถ้าในชื่อไม่มีเดือนอยู่เลย ให้ต่อท้ายชื่อไปเลย
    if target_date not in current_name:
        return f"{current_name}_{target_date}"
        
    return current_name

# ==========================================
# 🚀 ฟังก์ชันการทำงานหลัก
# ==========================================

def process_google_sheets_folder(folder_id, update_sort_on=True, download_on=True):
    json_files = [f for f in os.listdir('.') if f.endswith('.json')]
    if not json_files:
        print("❌ Error: ไม่พบไฟล์ .json")
        return
    key_file = json_files[0]

    try:
        scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
        creds = ServiceAccountCredentials.from_json_keyfile_name(key_file, scope)
        client = gspread.authorize(creds)
        drive_service = build('drive', 'v3', credentials=creds)
    except Exception as e:
        print(f"❌ การยืนยันตัวตนล้มเหลว: {e}")
        return

    if download_on and not os.path.exists(DOWNLOAD_FOLDER_NAME):
        os.makedirs(DOWNLOAD_FOLDER_NAME)

    try:
        query = f"'{folder_id}' in parents and mimeType = 'application/vnd.google-apps.spreadsheet' and trashed = false"
        results = drive_service.files().list(q=query, fields="files(id, name)").execute()
        files_in_folder = results.get('files', [])

        if not files_in_folder:
            print(f"⚠️ ไม่พบไฟล์ใน Folder ID: {folder_id}")
            return

        for f in files_in_folder:
            file_id = f['id']
            old_name = f['name']
            
            # --- ส่วนที่เพิ่ม: เปลี่ยนชื่อไฟล์ตามเดือนปัจจุบัน (รักษา ID เดิม) ---
            new_name = generate_new_filename(old_name)
            if old_name != new_name:
                print(f"\n🔄 กำลังเปลี่ยนชื่อไฟล์: {old_name} ➡️ {new_name}")
                drive_service.files().update(fileId=file_id, body={'name': new_name}).execute()
                current_file_name = new_name
            else:
                current_file_name = old_name

            print(f"\n🚀 จัดการไฟล์: {current_file_name}")
            
            # --- ขั้นตอนที่ 1 & 2: แก้ไขข้อมูลและจัดเรียงลำดับชีท (ตามตรรกะเดิม) ---
            if update_sort_on:
                try:
                    spreadsheet = client.open_by_key(file_id)
                    all_worksheets = spreadsheet.worksheets()
                    print(f"  - จัดเรียงชีท...")
                    sorted_sheets = sorted(all_worksheets, key=lambda x: x.title)
                    spreadsheet.reorder_worksheets(sorted_sheets)

                    for sheet in sorted_sheets:
                        sheet.update_acell('A1', FORMULA_A1)
                        print(f"    ✅ อัปเดต A1: {sheet.title}")
                        time.sleep(0.5)
                except Exception as e:
                    print(f"    ❌ แก้ไขไฟล์ไม่ได้: {e}")

            # --- ขั้นตอนที่ 3: ดาวน์โหลดไฟล์ (ตามตรรกะเดิม) ---
            if download_on:
                print(f"  - ดาวน์โหลด Excel...")
                try:
                    request = drive_service.files().export_media(
                        fileId=file_id,
                        mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
                    )
                    save_path = os.path.join(DOWNLOAD_FOLDER_NAME, f"{current_file_name}.xlsx")
                    with io.FileIO(save_path, 'wb') as fh:
                        downloader = MediaIoBaseDownload(fh, request)
                        done = False
                        while not done:
                            status, done = downloader.next_chunk()
                    print(f"  ✅ สำเร็จ -> {save_path}")
                except Exception as e:
                    print(f"    ❌ ดาวน์โหลดล้มเหลว: {e}")
                
                time.sleep(1)

    except Exception as e:
        print(f"❌ Error Folder {folder_id}: {e}")

# ==========================================
# 🚀 Execution
# ==========================================
if __name__ == "__main__":
    print("="*40)
    print("🤖 เริ่มระบบจัดการไฟล์อัตโนมัติ...")
    print("="*40)
    
    for folder_id in FOLDER_LIST:
        print(f"\n📂 ตรวจสอบ Folder: {folder_id}")
        process_google_sheets_folder(
            folder_id=folder_id, 
            update_sort_on=update_sort_set, # เปลี่ยนเป็น True ถ้าต้องการเรียงชีทและแก้ A1
            download_on= download_set
        )
        time.sleep(3)

    print("\n" + "="*40)
    print("🎯 ดำเนินการเสร็จสิ้น!")
    print("="*40)

🤖 เริ่มระบบจัดการไฟล์อัตโนมัติ...

📂 ตรวจสอบ Folder: 1BInVYvIoKW9jpG135ubNTlLdEakR1IKG



🚀 จัดการไฟล์: ไฟล์ราคากลาง ชุด บ_มี.ค. 69_
  - ดาวน์โหลด Excel...


  ✅ สำเร็จ -> ราคากลาง\ไฟล์ราคากลาง ชุด บ_มี.ค. 69_.xlsx



🚀 จัดการไฟล์: ไฟล์ราคากลาง ชุด บ1_มี.ค. 69_
  - ดาวน์โหลด Excel...


  ✅ สำเร็จ -> ราคากลาง\ไฟล์ราคากลาง ชุด บ1_มี.ค. 69_.xlsx



🚀 จัดการไฟล์: ไฟล์ราคากลาง ชุด อ_มี.ค. 69_
  - ดาวน์โหลด Excel...


  ✅ สำเร็จ -> ราคากลาง\ไฟล์ราคากลาง ชุด อ_มี.ค. 69_.xlsx



🚀 จัดการไฟล์: ไฟล์ราคากลาง ชุด น_มี.ค. 69_
  - ดาวน์โหลด Excel...


  ✅ สำเร็จ -> ราคากลาง\ไฟล์ราคากลาง ชุด น_มี.ค. 69_.xlsx



🎯 ดำเนินการเสร็จสิ้น!


In [3]:
# pip install gspread oauth2client

In [4]:
# import gspread
# from oauth2client.service_account import ServiceAccountCredentials
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaIoBaseDownload
# import os
# import io
# import time
# from datetime import datetime
# import re

# # ==========================================
# # ⚙️ ตั้งค่าเริ่มต้น (Configuration)
# # ==========================================
# FOLDER_LIST = [
#     "1BInVYvIoKW9jpG135ubNTlLdEakR1IKG"
# ]

# DOWNLOAD_FOLDER_NAME = "ราคากลาง"
# FORMULA_A1 = '=TEXT(TODAY(), "[$-th-TH]mmm e")'

# # # ตัวเลือกการทำงาน
# # update_sort_set = True  # จัดเรียงชีทและอัปเดตช่อง A1
# # download_set = True     # ดาวน์โหลดไฟล์ลงเครื่อง

# # ==========================================
# # 🛠️ ฟังก์ชันช่วยเหลือสำหรับจัดการชื่อไฟล์
# # ==========================================

# def get_thai_month_year():
#     """สร้างข้อความเดือนย่อ และ ปี พ.ศ. สองหลัก เช่น 'มี.ค. 69'"""
#     thai_months = ["", "ม.ค.", "ก.พ.", "มี.ค.", "เม.ย.", "พ.ค.", "มิ.ย.", "ก.ค.", "ส.ค.", "ก.ย.", "ต.ค.", "พ.ย.", "ธ.ค."]
#     now = datetime.now()
#     month_str = thai_months[now.month]
#     year_str = str(now.year + 543)[-2:]
#     return f"{month_str} {year_str}"

# def generate_new_filename(current_name):
#     """เปลี่ยนเดือน/ปี ในชื่อไฟล์อัตโนมัติ โดยใช้ Regex (รองรับ 'ชุด อ' และชื่อทุกรูปแบบ)"""
#     target_date = get_thai_month_year()
    
#     # รูปแบบ: ค้นหาเดือนย่อไทย ตามด้วยช่องว่าง(ถ้ามี) และปี 2 หลัก เช่น 'ก.พ. 68' หรือ 'ม.ค. 69'
#     pattern = r"(ม\.ค\.|ก\.พ\.|มี\.ค\.|เม\.ย\.|พ\.ค\.|มิ\.ย\.|ก\.ค\.|ส\.ค\.|ก\.ย\.|ต\.ค\.|พ\.ย\.|ธ\.ค\.)\s?\d{2}"
    
#     if re.search(pattern, current_name):
#         # ถ้าพบเดือนเก่าในชื่อ ให้แทนที่ด้วยเดือนปัจจุบันทันที
#         new_name = re.sub(pattern, target_date, current_name)
#         return new_name
    
#     # ถ้าในชื่อไม่มีรูปแบบเดือนอยู่เลย ให้เติมต่อท้ายชื่อเดิม
#     if target_date not in current_name:
#         return f"{current_name}_{target_date}"
        
#     return current_name

# # ==========================================
# # 🚀 ฟังก์ชันการทำงานหลัก
# # ==========================================

# def process_google_sheets_folder(folder_id, update_sort_on=True, download_on=True):
#     # ค้นหาไฟล์ JSON สำหรับยืนยันตัวตน
#     json_files = [f for f in os.listdir('.') if f.endswith('.json')]
#     if not json_files:
#         print("❌ Error: ไม่พบไฟล์ .json ในโฟลเดอร์ปัจจุบัน")
#         return
#     key_file = json_files[0]

#     try:
#         scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
#         creds = ServiceAccountCredentials.from_json_keyfile_name(key_file, scope)
#         client = gspread.authorize(creds)
#         drive_service = build('drive', 'v3', credentials=creds)
#     except Exception as e:
#         print(f"❌ การยืนยันตัวตนล้มเหลว: {e}")
#         return

#     if download_on and not os.path.exists(DOWNLOAD_FOLDER_NAME):
#         os.makedirs(DOWNLOAD_FOLDER_NAME)

#     try:
#         # ค้นหาเฉพาะ Google Sheets ที่ไม่ได้อยู่ในถังขยะ
#         query = f"'{folder_id}' in parents and mimeType = 'application/vnd.google-apps.spreadsheet' and trashed = false"
#         results = drive_service.files().list(q=query, fields="files(id, name)").execute()
#         files_in_folder = results.get('files', [])

#         if not files_in_folder:
#             print(f"⚠️ ไม่พบไฟล์ Google Sheets ใน Folder ID: {folder_id}")
#             return

#         for f in files_in_folder:
#             file_id = f['id']
#             old_name = f['name']
            
#             # 1. เปลี่ยนชื่อไฟล์บน Cloud (ถ้าจำเป็น)
#             new_name = generate_new_filename(old_name)
#             if old_name != new_name:
#                 print(f"\n🔄 เปลี่ยนชื่อ: {old_name} ➡️ {new_name}")
#                 drive_service.files().update(fileId=file_id, body={'name': new_name}).execute()
#                 current_file_name = new_name
#             else:
#                 current_file_name = old_name

#             print(f"\n🚀 จัดการไฟล์: {current_file_name}")
            
#             # 2. จัดเรียงชีทและอัปเดตสูตรใน A1
#             if update_sort_on:
#                 try:
#                     spreadsheet = client.open_by_key(file_id)
#                     all_worksheets = spreadsheet.worksheets()
#                     print(f"  - กำลังจัดเรียงชีทตามชื่อ...")
#                     sorted_sheets = sorted(all_worksheets, key=lambda x: x.title)
#                     spreadsheet.reorder_worksheets(sorted_sheets)

#                     for sheet in sorted_sheets:
#                         sheet.update_acell('A1', FORMULA_A1)
#                         print(f"    ✅ อัปเดต A1: {sheet.title}")
#                         time.sleep(0.5) # ป้องกันการเรียกใช้ API ถี่เกินไป
#                 except Exception as e:
#                     print(f"    ❌ แก้ไขข้อมูลชีทไม่ได้: {e}")

#             # 3. ดาวน์โหลดเป็นไฟล์ Excel (.xlsx)
#             if download_on:
#                 print(f"  - กำลังดาวน์โหลดเป็น Excel...")
#                 try:
#                     request = drive_service.files().export_media(
#                         fileId=file_id,
#                         mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
#                     )
#                     save_path = os.path.join(DOWNLOAD_FOLDER_NAME, f"{current_file_name}.xlsx")
#                     with io.FileIO(save_path, 'wb') as fh:
#                         downloader = MediaIoBaseDownload(fh, request)
#                         done = False
#                         while not done:
#                             status, done = downloader.next_chunk()
#                     print(f"  ✅ ดาวน์โหลดสำเร็จ -> {save_path}")
#                 except Exception as e:
#                     print(f"    ❌ ดาวน์โหลดล้มเหลว: {e}")
                
#                 time.sleep(1)

#     except Exception as e:
#         print(f"❌ Error Folder {folder_id}: {e}")

# # ==========================================
# # 🚀 Execution
# # ==========================================
# if __name__ == "__main__":
#     print("="*40)
#     print("🤖 เริ่มระบบจัดการไฟล์อัตโนมัติ...")
#     print("="*40)
    
#     for folder_id in FOLDER_LIST:
#         print(f"\n📂 ตรวจสอบ Folder: {folder_id}")
#         process_google_sheets_folder(
#             folder_id=folder_id, 
#             update_sort_on=update_sort_set, 
#             download_on=download_set
#         )
#         time.sleep(2)

#     print("\n" + "="*40)
#     print("🎯 ดำเนินการทุกอย่างเสร็จสิ้น!")
#     print("="*40)

In [5]:
# if __name__ == "__main__":
#     print("="*40)
#     print("🤖 เริ่มระบบจัดการไฟล์อัตโนมัติ...")
#     print("="*40)
    
#     for folder_id in FOLDER_LIST:
#         print(f"\n📂 ตรวจสอบ Folder: {folder_id}")
#         # --- แก้ไขจุดที่ 2: ตรวจสอบให้แน่ใจว่าชื่อตัวแปรตรงกัน ---
#         process_google_sheets_folder(
#             folder_id=folder_id, 
#             update_sort_on=update_sort_set, 
#             download_on=download_set
#         )
#         time.sleep(2)

#     print("\n" + "="*40)
#     print("🎯 ดำเนินการทุกอย่างเสร็จสิ้น!")
#     print("="*40)

# ยังไม่ใช้

In [6]:
# import gspread
# from oauth2client.service_account import ServiceAccountCredentials
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaIoBaseDownload
# import os
# import io
# import time

# # ==========================================
# # ⚙️ ตั้งค่าเริ่มต้น (Configuration)
# # ==========================================
# FOLDER_LIST = [
#     "1BInVYvIoKW9jpG135ubNTlLdEakR1IKG",
# ]

# DOWNLOAD_FOLDER_NAME = "ราคากลาง_Excel"

# # ==========================================
# # 🛠️ ฟังก์ชันสำหรับดาวน์โหลดเฉพาะ Excel (.xlsx)
# # ==========================================

# def download_only_excel_files(folder_id):
#     # 1. ค้นหาไฟล์ JSON Key
#     json_files = [f for f in os.listdir('.') if f.endswith('.json')]
#     if not json_files:
#         print("❌ Error: ไม่พบไฟล์ .json")
#         return
#     key_file = json_files[0]

#     # 2. ยืนยันตัวตน
#     creds = ServiceAccountCredentials.from_json_keyfile_name(key_file, ["https://www.googleapis.com/auth/drive"])
#     drive_service = build('drive', 'v3', credentials=creds)

#     if not os.path.exists(DOWNLOAD_FOLDER_NAME):
#         os.makedirs(DOWNLOAD_FOLDER_NAME)

#     try:
#         # 3. Query เฉพาะไฟล์ที่มี MimeType เป็น Excel (.xlsx) เท่านั้น
#         query = f"'{folder_id}' in parents and mimeType = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet' and trashed = false"
#         results = drive_service.files().list(q=query, fields="files(id, name)").execute()
#         files_in_folder = results.get('files', [])

#         if not files_in_folder:
#             print(f"⚠️ ไม่พบไฟล์ .xlsx ใน Folder ID: {folder_id}")
#             return

#         for f in files_in_folder:
#             file_id = f['id']
#             file_name = f['name']
#             print(f"\n🚀 กำลังโหลดไฟล์ Excel: {file_name}")
            
#             # 4. ดาวน์โหลดไฟล์ต้นฉบับ (get_media)
#             try:
#                 request = drive_service.files().get_media(fileId=file_id)
#                 save_path = os.path.join(DOWNLOAD_FOLDER_NAME, file_name)
                
#                 with io.FileIO(save_path, 'wb') as fh:
#                     downloader = MediaIoBaseDownload(fh, request)
#                     done = False
#                     while not done:
#                         status, done = downloader.next_chunk()
#                 print(f"  ✅ ดาวน์โหลดสำเร็จ -> {save_path}")
                
#             except Exception as e:
#                 print(f"  ❌ ดาวน์โหลดล้มเหลว: {e}")
            
#             time.sleep(1) # พัก 1 วินาทีป้องกัน API Limit

#     except Exception as e:
#         print(f"❌ เกิดข้อผิดพลาดกับ Folder {folder_id}: {e}")

# # ==========================================
# # 🚀 เริ่มการทำงาน
# # ==========================================
# if __name__ == "__main__":
#     print("🤖 เริ่มระบบดึงไฟล์ Excel จาก Google Drive...")
#     for folder_id in FOLDER_LIST:
#         print(f"\n📂 ตรวจสอบ Folder: {folder_id}")
#         download_only_excel_files(folder_id)
#         time.sleep(2)
#     print("\n🎯 เสร็จสิ้นการดาวน์โหลด Excel ทั้งหมด!")

In [7]:
# pip install gspread oauth2client

In [8]:
# import os
# import requests
# import openpyxl
# from datetime import datetime
# from copy import copy
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaFileUpload
# from google.oauth2 import service_account

# # ==========================================
# # 0. ตั้งค่าวันที่และเดือนภาษาไทย
# # ==========================================
# thai_months = [
#     "มกราคม", "กุมภาพันธ์", "มีนาคม", "เมษายน", "พฤษภาคม", "มิถุนายน",
#     "กรกฎาคม", "สิงหาคม", "กันยายน", "ตุลาคม", "พฤศจิกายน", "ธันวาคม"
# ]

# now = datetime.now()
# month_thai = thai_months[now.month - 1]
# year_thai = now.year + 543  # แปลงเป็น พ.ศ.

# # รูปแบบชื่อที่จะใช้ (เช่น: กุมภาพันธ์ 2569)
# name_date = f"{month_thai} {year_thai}"

# # ==========================================
# # 1. ตั้งค่าข้อมูลพื้นฐาน
# # ==========================================
# SOURCE_SPREADSHEET_ID = "1R5QjxWWMw7SYsPSblzNzkJl_nKEsYl81" # testfile
# SOURCE_SPREADSHEET_ID = "1K6SpCFJmOHZDC1P2rdZG-WjnlLYrKGA3" # testfile
# EXISTING_FILE_ID = '1K6SpCFJmOHZDC1P2rdZG-WjnlLYrKGA3'

# # ชื่อไฟล์ใหม่ (เช่น ราคากลาง ชุด บ กุมภาพันธ์ 2569.xlsx)
# NEW_FILE_NAME = f"ราคากลาง ชุด บ อะไร {name_date}.xlsx"

# current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
# input_file = os.path.join(current_dir, "full_workbook.xlsx")
# output_file = os.path.join(current_dir, NEW_FILE_NAME)
# credentials_path = os.path.join(current_dir, "service_account.json")

# # ==========================================
# # 2. ฟังก์ชันดาวน์โหลด (Download)
# # ==========================================
# def download_file():
#     excel_url = f"https://docs.google.com/spreadsheets/d/{SOURCE_SPREADSHEET_ID}/export?format=xlsx"
#     print(f"1. กำลังดาวน์โหลดจาก Sheets ID: {SOURCE_SPREADSHEET_ID}")
#     response = requests.get(excel_url)
#     if response.status_code == 200:
#         with open(input_file, 'wb') as f:
#             f.write(response.content)
#         print(f"✅ ดาวน์โหลดสำเร็จ")
#     else:
#         raise Exception("ดาวน์โหลดไม่สำเร็จ ตรวจสอบการแชร์ไฟล์ต้นทาง")

# # ==========================================
# # 3. ฟังก์ชันแก้ไขและรักษา Format (Edit)
# # ==========================================
# def edit_excel_safely():
#     print(f"2. กำลังแก้ไขไฟล์และรักษา Format สีสัน...")
#     wb = openpyxl.load_workbook(input_file)
    
#     for sheet_name in wb.sheetnames:
#         ws = wb[sheet_name]
#         target = ws['A1']
        
#         # ก๊อปปี้รูปแบบเดิม
#         old_font = copy(target.font)
#         old_fill = copy(target.fill)
#         old_alignment = copy(target.alignment)
#         old_border = copy(target.border)
        
#         # ใส่ค่าใหม่เป็น เดือน พ.ศ. (เช่น กุมภาพันธ์ 2569)
#         target.value = name_date
        
#         # แปะรูปแบบเดิมกลับคืน
#         target.font = old_font
#         target.fill = old_fill
#         target.alignment = old_alignment
#         target.border = old_border
        
#         print(f" - แก้ไขชีท [{sheet_name}] เรียบร้อย")
        
#     wb.save(output_file)
#     print(f"✅ เซฟไฟล์ในเครื่องชื่อ: {NEW_FILE_NAME}")

# # ==========================================
# # 4. ฟังก์ชันอัปเดตและเปลี่ยนชื่อบน Drive (Update)
# # ==========================================
# def update_to_drive():
#     print("3. กำลังเชื่อมต่อ Google Drive เพื่ออัปเดตข้อมูล...")
#     if not os.path.exists(credentials_path):
#         print(f"❌ ไม่พบไฟล์ credentials.json ที่: {credentials_path}")
#         return

#     creds = service_account.Credentials.from_service_account_file(
#         credentials_path, 
#         scopes=['https://www.googleapis.com/auth/drive']
#     )
#     service = build('drive', 'v3', credentials=creds)

#     file_metadata = {
#         'name': NEW_FILE_NAME 
#     }

#     media = MediaFileUpload(
#         output_file, 
#         mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
#         resumable=True
#     )
    
#     print(f" - กำลังเขียนทับไฟล์ ID: {EXISTING_FILE_ID}")
    
#     file = service.files().update(
#         fileId=EXISTING_FILE_ID,
#         body=file_metadata,
#         media_body=media,
#         supportsAllDrives=True 
#     ).execute()
    
#     print(f"✅ อัปเดตและเปลี่ยนชื่อบน Drive เป็น '{NEW_FILE_NAME}' สำเร็จ!")

# if __name__ == "__main__":
#     try:
#         download_file()
#         edit_excel_safely()
#         update_to_drive()
#         print("\n🎉 ทุกขั้นตอนเสร็จสมบูรณ์!")
#     except Exception as e:
#         print(f"\n❌ เกิดข้อผิดพลาด: {e}")

In [9]:
# import os
# import requests
# import openpyxl
# from datetime import datetime
# from copy import copy
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaFileUpload
# from google.oauth2 import service_account

# # ==========================================
# # 0. ตั้งค่าวันที่และเดือนภาษาไทย
# # ==========================================
# thai_months = [
#     "มกราคม", "กุมภาพันธ์", "มีนาคม", "เมษายน", "พฤษภาคม", "มิถุนายน",
#     "กรกฎาคม", "สิงหาคม", "กันยายน", "ตุลาคม", "พฤศจิกายน", "ธันวาคม"
# ]

# now = datetime.now()
# month_thai = thai_months[now.month - 1]
# year_thai = now.year + 543 

# name_date = f"{month_thai} {year_thai}"

# # ==========================================
# # 1. ตั้งค่าข้อมูลพื้นฐาน
# # ==========================================
# SOURCE_SPREADSHEET_ID = "1K6SpCFJmOHZDC1P2rdZG-WjnlLYrKGA3"
# EXISTING_FILE_ID = '1K6SpCFJmOHZDC1P2rdZG-WjnlLYrKGA3'
# NEW_FILE_NAME = f"ราคากลาง ชุด บ อะไร {name_date}.xlsx"

# current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
# input_file = os.path.join(current_dir, "full_workbook.xlsx")
# output_file = os.path.join(current_dir, NEW_FILE_NAME)
# credentials_path = os.path.join(current_dir, "service_account.json")

# # ==========================================
# # 2. ฟังก์ชันดาวน์โหลด (Download)
# # ==========================================
# def download_file():
#     excel_url = f"https://docs.google.com/spreadsheets/d/{SOURCE_SPREADSHEET_ID}/export?format=xlsx"
#     print(f"1. กำลังดาวน์โหลดจาก Sheets ID: {SOURCE_SPREADSHEET_ID}")
#     response = requests.get(excel_url)
#     if response.status_code == 200:
#         with open(input_file, 'wb') as f:
#             f.write(response.content)
#         print(f"✅ ดาวน์โหลดสำเร็จ")
#     else:
#         raise Exception("ดาวน์โหลดไม่สำเร็จ ตรวจสอบการแชร์ไฟล์ต้นทาง")

# # ==========================================
# # 3. ฟังก์ชันแก้ไขแบบ "ไม่แตะต้องสูตร" (Edit)
# # ==========================================
# def edit_excel_safely():
#     print(f"2. กำลังแก้ไขไฟล์ (ข้ามเซลล์ที่มีสูตร)...")
#     # ห้ามใส่ data_only=True เพราะจะทำให้สูตรในไฟล์หายกลายเป็นค่าคงที่
#     wb = openpyxl.load_workbook(input_file)
    
#     for sheet_name in wb.sheetnames:
#         ws = wb[sheet_name]
#         target = ws['A1']
        
#         # ตรวจสอบว่าเป็นสูตรหรือไม่ (f = formula)
#         if target.data_type == 'f':
#             print(f" ⚠️  ข้ามชีท [{sheet_name}]: เซลล์ A1 มีสูตรอยู่แล้ว ({target.value})")
#             continue
            
#         # ถ้าไม่ใช่สูตร ถึงจะทำการแก้ไข
#         print(f" 📝 แก้ไขชีท [{sheet_name}]...")
        
#         # ก๊อปปี้รูปแบบเดิม
#         old_font = copy(target.font)
#         old_fill = copy(target.fill)
#         old_alignment = copy(target.alignment)
#         old_border = copy(target.border)
        
#         # ใส่ค่าใหม่
#         target.value = name_date
        
#         # แปะรูปแบบเดิมกลับคืน
#         target.font = old_font
#         target.fill = old_fill
#         target.alignment = old_alignment
#         target.border = old_border
        
#     wb.save(output_file)
#     print(f"✅ เซฟไฟล์ลงเครื่องเรียบร้อย: {NEW_FILE_NAME}")

# # ==========================================
# # 4. ฟังก์ชันอัปเดตและเปลี่ยนชื่อบน Drive (Update)
# # ==========================================
# def update_to_drive():
#     print("3. กำลังเชื่อมต่อ Google Drive...")
#     if not os.path.exists(credentials_path):
#         print(f"❌ ไม่พบไฟล์ credentials.json ที่: {credentials_path}")
#         return

#     creds = service_account.Credentials.from_service_account_file(
#         credentials_path, 
#         scopes=['https://www.googleapis.com/auth/drive']
#     )
#     service = build('drive', 'v3', credentials=creds)

#     file_metadata = { 'name': NEW_FILE_NAME }
#     media = MediaFileUpload(
#         output_file, 
#         mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
#         resumable=True
#     )
    
#     file = service.files().update(
#         fileId=EXISTING_FILE_ID,
#         body=file_metadata,
#         media_body=media,
#         supportsAllDrives=True 
#     ).execute()
    
#     print(f"✅ อัปเดตและเปลี่ยนชื่อบน Drive เรียบร้อย!")

# if __name__ == "__main__":
#     try:
#         download_file()
#         edit_excel_safely()
#         update_to_drive()
#         print("\n🎉 ทุกขั้นตอนเสร็จสมบูรณ์!")
#     except Exception as e:
#         print(f"\n❌ เกิดข้อผิดพลาด: {e}")